# 01_Data Inspection

This notebook performs the initial inspection of all raw datasets before any preprocessing. The objective is to understand the structure, quality, and completeness of each dataset.

In [1]:
from __future__ import annotations

from pathlib import Path
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

RANDOM_STATE = 42
PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"

pd.set_option("display.max_columns", None)

## Loading Utility Functions

These reusable functions load datasets and generate a concise summary including shape, memory usage, data types, and descriptive statistics.

In [2]:
def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    logging.info("Loading %s", path.name)
    return pd.read_csv(path)


def summarize(df: pd.DataFrame, name: str) -> None:
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(f"Shape: {df.shape}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    display(df.head())
    display(df.dtypes.to_frame("dtype"))
    display(df.describe(include="all").T)

## Data Quality Utility Functions

These helper functions evaluate missing values, uniqueness, duplicates, and perform simple numeric validation across all datasets.

In [3]:
def quality_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
        "duplicates": df.duplicated().sum(),
    })
    return report


def validate_numeric(df: pd.DataFrame) -> dict[str, str]:
    issues = {}
    for c in df.select_dtypes(include="number"):
        if (df[c] < 0).any():
            issues[c] = "Contains negative values"
    return issues

## Load Raw Datasets

Locate all CSV files inside the raw data directory and load them into a dictionary for subsequent inspection.

In [4]:
raw_files = sorted(RAW_DATA.glob("*.csv"))
assert len(raw_files) == 8, f"Expected 8 CSV files in {RAW_DATA}, found {len(raw_files)}"

datasets = {path.stem: load_csv(path) for path in raw_files}
dataset_names = list(datasets)
print(f"Loaded {len(datasets)} datasets: {', '.join(dataset_names)}")

2026-07-19 19:05:29,552 | INFO | Loading customers.csv
2026-07-19 19:05:34,353 | INFO | Loading geolocation.csv
2026-07-19 19:05:34,374 | INFO | Loading order_items.csv
2026-07-19 19:05:41,563 | INFO | Loading order_payments.csv
2026-07-19 19:05:43,691 | INFO | Loading order_reviews.csv
2026-07-19 19:05:49,029 | INFO | Loading orders.csv
2026-07-19 19:05:57,391 | INFO | Loading products.csv
2026-07-19 19:05:57,406 | INFO | Loading sellers.csv


Loaded 8 datasets: customers, geolocation, order_items, order_payments, order_reviews, orders, products, sellers


## Basic Dataset Inspection

Review the structure and summary statistics of every dataset before performing any cleaning.

In [5]:
for name, df in datasets.items():
    summarize(df, name)

customers
Shape: (1000000, 9)
Memory: 441.81 MB


,customer_id,customer_unique_id,customer_name,customer_gender,customer_age,customer_zip_code_prefix,customer_city,customer_state,customer_segment
0,1e2e2881-a0eb-4cb0-829f-a566e810d05f,d45cdff2-5195-41e2-a0e5-6fe597e378dd,James Ramos,M,47,85037,Phoenix,AZ,Consumer
1,380b7418-308c-4bf7-b2bd-3ee446cb9ea6,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,M,18,90060,Los Angeles,CA,Consumer
2,89b0d980-868f-478d-a63b-5fea5f265f4f,c820d3a4-edd4-4b32-9605-22d97dcc8e5e,Debra Jones,F,61,30398,Atlanta,GA,Consumer
3,336d14e6-a22f-4126-b2b7-fa64a2955b34,d3d86bff-2f03-4bec-9cbf-9a2f8f0d5783,Jamie Estrada,F,34,30352,Atlanta,GA,Consumer
4,340c9f9d-bb72-4bfa-a6aa-aa9c7c18a03f,35cee471-325e-4ad2-8e4e-7b169dc6df81,Brian Jackson,M,18,90060,Los Angeles,CA,Consumer


,dtype
customer_id,object
customer_unique_id,object
customer_name,object
customer_gender,object
customer_age,int64
customer_zip_code_prefix,int64
customer_city,object
customer_state,object
customer_segment,object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,1000000,1000000,1e2e2881-a0eb-4cb0-829f-a566e810d05f,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,1000000,279199,f564577f-8fff-4fa9-bfba-9d710170cfec,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_name,1000000,152603,Jennifer Smith,416,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_gender,1000000,2,F,651080,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_age,1000000.0,NaN,NaN,NaN,35.383124,11.219629,18.0,27.0,35.0,43.0,70.0
customer_zip_code_prefix,1000000.0,NaN,NaN,NaN,57314.170948,30624.162507,10010.0,30369.0,77012.0,90019.0,98199.0
customer_city,1000000,10,Los Angeles,251542,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,1000000,10,CA,251542,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_segment,1000000,3,Consumer,700676,NaN,NaN,NaN,NaN,NaN,NaN,NaN


geolocation
Shape: (11500, 5)
Memory: 1.45 MB


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,2000,2000,cdc4381f-fa67-41f2-a7d5-07f330097589,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_category_name,2000,7,books,310,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,2000,1570,Zenith Building Blocks Toy,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_brand,2000,8,Nova,257,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_weight_g,2000.0,NaN,NaN,NaN,6555.015,10777.216718,100.0,608.75,1424.0,6175.5,49869.0
product_length_cm,2000.0,NaN,NaN,NaN,48.9385,43.596801,5.0,19.0,33.0,62.0,199.0
product_height_cm,2000.0,NaN,NaN,NaN,16.7445,7.294095,5.0,10.0,17.0,23.0,29.0
product_width_cm,2000.0,NaN,NaN,NaN,16.81,7.085337,5.0,11.0,17.0,23.0,29.0
cost,2000.0,NaN,NaN,NaN,223.64733,339.807816,4.43,23.5525,73.47,275.7475,1693.12
price,2000.0,NaN,NaN,NaN,322.65161,419.82025,10.01,45.4875,150.925,404.5875,1999.85


sellers
Shape: (500, 8)
Memory: 0.18 MB


,seller_id,seller_company_name,seller_contact_name,seller_contact_gender,seller_contact_age,seller_zip_code_prefix,seller_city,seller_state
0,82787b9a-3c5a-476a-9f1b-a5fc7d98c8cf,Johnson LLC,Holly Mcdaniel,F,57,60633,Chicago,IL
1,75c29e4b-cc77-45c9-a3b6-7aa713f56126,Hicks LLC,Charles Casey,M,33,30339,Atlanta,GA
2,d2bc11e3-2a8f-4d16-9b04-8db5008730de,Briggs-Brown,Daniel Serrano,M,33,98193,Seattle,WA
3,aacb0fbc-8983-41b2-b9df-e2c35b8ff171,Flores LLC,Dillon Brooks Jr.,M,30,90056,Los Angeles,CA
4,098ae009-ca72-4820-8e53-6a279fa26e63,Sosa Ltd,Vanessa Stone,F,46,60615,Chicago,IL


,dtype
seller_id,object
seller_company_name,object
seller_contact_name,object
seller_contact_gender,object
seller_contact_age,int64
seller_zip_code_prefix,int64
seller_city,object
seller_state,object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
seller_id,500,500,82787b9a-3c5a-476a-9f1b-a5fc7d98c8cf,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_company_name,500,493,Johnson Ltd,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_contact_name,500,500,Holly Mcdaniel,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_contact_gender,500,2,F,250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_contact_age,500.0,NaN,NaN,NaN,35.938,11.363116,18.0,27.0,36.0,44.0,70.0
seller_zip_code_prefix,500.0,NaN,NaN,NaN,58796.928,31977.469657,10011.0,30395.0,77015.0,90031.25,98198.0
seller_city,500,10,Los Angeles,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_state,500,10,CA,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Duplicate Record Analysis

Identify duplicate records in each dataset to assess initial data quality.

In [6]:
print("\n================ DUPLICATE RECORDS ================\n")

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name} --> Duplicate Rows: {duplicate_count}")


================ DUPLICATE RECORDS ================

customers --> Duplicate Rows: 0
geolocation --> Duplicate Rows: 0
order_items --> Duplicate Rows: 0
order_payments --> Duplicate Rows: 0
order_reviews --> Duplicate Rows: 0
orders --> Duplicate Rows: 0
products --> Duplicate Rows: 0
sellers --> Duplicate Rows: 0


## Data Quality Assessment

Generate quality reports and validate numeric columns for potential anomalies.

In [7]:
quality_reports = {name: quality_report(df) for name, df in datasets.items()}

for name, report in quality_reports.items():
    print(f"\nQuality report: {name}")
    display(report.sort_values(["missing_%", "unique"], ascending=[False, False]))

numeric_issues = {name: validate_numeric(df) for name, df in datasets.items()}
numeric_issues = {name: issues for name, issues in numeric_issues.items() if issues}

print("Numeric validation issues:")
display(pd.Series(numeric_issues, name="issues").to_frame()) if numeric_issues else print("None found")


Quality report: customers


,dtype,missing,missing_%,unique,duplicates
customer_id,object,0,0.0,1000000,0
customer_unique_id,object,0,0.0,279199,0
customer_name,object,0,0.0,152603,0
customer_zip_code_prefix,int64,0,0.0,900,0
customer_age,int64,0,0.0,53,0
customer_city,object,0,0.0,10,0
customer_state,object,0,0.0,10,0
customer_segment,object,0,0.0,3,0
customer_gender,object,0,0.0,2,0



Quality report: geolocation


,dtype,missing,missing_%,unique,duplicates
geolocation_lat,float64,0,0.0,11500,0
geolocation_lng,float64,0,0.0,11500,0
zip_code_prefix,int64,0,0.0,900,0
geolocation_city,object,0,0.0,10,0
geolocation_state,object,0,0.0,10,0



Quality report: order_items


...

,dtype,missing,missing_%,unique,duplicates
product_id,object,0,0.0,2000,0
price,float64,0,0.0,1946,0
cost,float64,0,0.0,1900,0
product_weight_g,int64,0,0.0,1640,0
product_name,object,0,0.0,1570,0
product_length_cm,int64,0,0.0,188,0
product_height_cm,int64,0,0.0,25,0
product_width_cm,int64,0,0.0,25,0
product_brand,object,0,0.0,8,0
product_category_name,object,0,0.0,7,0



Quality report: sellers


,dtype,missing,missing_%,unique,duplicates
seller_id,object,0,0.0,500,0
seller_contact_name,object,0,0.0,500,0
seller_company_name,object,0,0.0,493,0
seller_zip_code_prefix,int64,0,0.0,361,0
seller_contact_age,int64,0,0.0,48,0
seller_city,object,0,0.0,10,0
seller_state,object,0,0.0,10,0
seller_contact_gender,object,0,0.0,2,0


Numeric validation issues:


,issues
geolocation,{'geolocation_lng': 'Contains negative values'}


## Inspection Summary

Create a concise inference table summarizing missing values, duplicates, and columns requiring further preprocessing.

In [9]:
def inspection_inferences(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in datasets.items():
        missing_cells = int(df.isna().sum().sum())
        duplicate_rows = int(df.duplicated().sum())
        object_columns = df.select_dtypes(include="object").columns.tolist()
        high_missing = quality_report(df).loc[lambda report: report["missing"] > 0].index.tolist()

        rows.append({
            "dataset": name,
            "inference": (
                f"{df.shape[0]:,} rows and {df.shape[1]} columns; "
                f"{missing_cells:,} missing cells across {len(high_missing)} columns; "
                f"{duplicate_rows:,} exact duplicate rows; "
                f"{len(object_columns)} text-like columns should be reviewed for whitespace and date parsing."
            ),
            "priority_columns": ", ".join(high_missing) if high_missing else "No missing values detected",
        })
    return pd.DataFrame(rows)

inspection_summary = inspection_inferences(datasets)
display(inspection_summary)

,dataset,inference,priority_columns
0,customers,"1,000,000 rows and 9 columns; 0 missing cells ...",No missing values detected
1,geolocation,"11,500 rows and 5 columns; 0 missing cells acr...",No missing values detected
2,order_items,"2,199,819 rows and 8 columns; 0 missing cells ...",No missing values detected
3,order_payments,"1,149,371 rows and 5 columns; 0 missing cells ...",No missing values detected
4,order_reviews,"933,748 rows and 7 columns; 0 missing cells ac...",No missing values detected
5,orders,"1,000,000 rows and 8 columns; 132,504 missing ...","order_delivered_carrier_date, order_delivered_..."
6,products,"2,000 rows and 10 columns; 0 missing cells acr...",No missing values detected
7,sellers,500 rows and 8 columns; 0 missing cells across...,No missing values detected


## Summary

The initial data inspection successfully loaded **8 datasets** containing customer, order, product, seller, payment, review, and geolocation information. Overall, the data quality is good, with **no duplicate records** and **no missing values** in seven of the eight datasets. The **orders** dataset is the only table containing missing values, specifically in the delivery-related timestamp columns, which is expected for orders that have not yet been delivered or were cancelled.

Several columns representing dates and timestamps are currently stored as **object** data types instead of **datetime**, which will limit time-series analysis and feature engineering if left unchanged. Numeric validation identified negative longitude values in the **geolocation** dataset; however, this is expected because longitude coordinates can legitimately be negative depending on geographic location.

Overall, the datasets are in good condition and require only minor preprocessing before exploratory data analysis (EDA) and feature engineering.

---

# Next Steps

### 1. Convert Date Columns to `datetime`
The following columns are currently stored as **object** and should be converted to `datetime` format:

**Orders**
- `order_purchase_timestamp`
- `order_approved_at`
- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- `order_estimated_delivery_date`

**Order Items**
- `shipping_limit_date`

**Order Reviews**
- `review_creation_date`
- `review_answer_timestamp`

---

### 2. Handle Missing Values

The **orders** dataset contains missing values in:

- `order_delivered_carrier_date`
- `order_delivered_customer_date`

These missing values likely correspond to undelivered or cancelled orders. Before imputing or removing them:

- Verify their relationship with `order_status`.
- Retain missing values if they are logically valid.
- Document the business reason for these missing records.

---

### 3. Validate Numeric Columns

Review numeric columns for invalid values such as:

- Negative prices
- Negative freight charges
- Negative product dimensions
- Invalid customer ages

**Note:** Negative values in `geolocation_lng` are expected geographical coordinates and do **not** require correction.

---

### 4. Verify Data Types

Ensure each feature has the appropriate data type:

| Data Type | Examples |
|-----------|----------|
| Datetime | Order and review timestamps |
| Integer | Age, Quantity, Installments |
| Float | Price, Cost, Freight |
| Category/Object | State, City, Payment Type, Product Category |

---

### 5. Perform Consistency Checks

Validate relationships across datasets:

- Every `customer_id` exists in the Customers table.
- Every `product_id` exists in the Products table.
- Every `seller_id` exists in the Sellers table.
- Every `order_id` exists across Orders, Payments, Reviews, and Order Items.

---

### 6. Detect Outliers

Inspect numeric variables such as:

- Product price
- Freight value
- Product weight
- Payment value
- Customer age

Apply visualization techniques (boxplots, histograms, IQR) to identify extreme values.

---

### 7. Prepare Clean Dataset

After validation:

- Convert data types.
- Handle missing values appropriately.
- Remove any invalid records if required.
- Standardize categorical values.
- Save the cleaned datasets to the `processed` data folder.

---

### 8. Proceed to Exploratory Data Analysis (EDA)

Once preprocessing is complete, continue with:

- Univariate Analysis
- Bivariate Analysis
- Correlation Analysis
- Customer Segmentation Analysis
- Sales Trend Analysis
- Product Performance Analysis
- Seller Performance Analysis
- Time-based Purchase Analysis
- Business Insights and Feature Engineering